## Ablation Study with LightGBM

**Cell 1: Imports**

In [16]:
import json
import time
from pathlib import Path

import numpy as np
import scipy.sparse as sp
import joblib
import lightgbm as lgb
from math import ceil

from lightgbm import LGBMClassifier

print("Imports loaded OK")


Imports loaded OK


**Cell 2: Check the folders to find the paths of Malware Project**

In [ ]:
print("Current PROJECT_ROOT:", PROJECT_ROOT)
print("\nParents:")
for i, p in enumerate(PROJECT_ROOT.parents):
    print(f"{i}: {p}")


Current PROJECT_ROOT: /Users/georgektenas/Desktop/Malware Project/code/MalDICT/LightGBM_Benchmark

Parents:
0: /Users/georgektenas/Desktop/Malware Project/code/MalDICT
1: /Users/georgektenas/Desktop/Malware Project/code
2: /Users/georgektenas/Desktop/Malware Project
3: /Users/georgektenas/Desktop
4: /Users/georgektenas
5: /Users
6: /


**Cell 3: Making the Data ROOT**

In [ ]:
ALWARE_PROJECT_ROOT = PROJECT_ROOT.parents[2]
DATA_ROOT = MALWARE_PROJECT_ROOT / "data" / "behavior_vectors_paper"

print("Malware project root:", MALWARE_PROJECT_ROOT)
print("Data root:", DATA_ROOT)

# Λίστα με μερικά .npz για να επιβεβαιώσουμε ότι τα βλέπει σωστά
npz_samples = list(DATA_ROOT.glob("*.npz"))[:20]
print(f"\nFound {len(list(DATA_ROOT.glob('*.npz')))} .npz files under DATA_ROOT")
for f in npz_samples:
    print(" -", f.name)


Malware project root: /Users/georgektenas/Desktop/Malware Project
Data root: /Users/georgektenas/Desktop/Malware Project/data/behavior_vectors_paper

Found 1192 .npz files under DATA_ROOT
 - train_part_capped298.npz
 - test_part040.npz
 - train_part259.npz
 - train_part_capped529.npz
 - train_part_capped501.npz
 - train_part265.npz
 - train_part503.npz
 - train_part_capped267.npz
 - train_part517.npz
 - train_part_capped273.npz
 - train_part_capped515.npz
 - train_part271.npz
 - train_part098.npz
 - train_part_capped059.npz
 - test_part_capped013.npz
 - test_part_capped007.npz
 - train_part_capped065.npz
 - train_part067.npz
 - train_part073.npz
 - train_part_capped071.npz


**Cell 4: Basic paths and lists train/test parts for X & Y**

In [ ]:
# Θα χρησιμοποιήσουμε ως VEC τον φάκελο behavior_vectors_paper
VEC = DATA_ROOT
BASE = MALWARE_PROJECT_ROOT

# Φόρτωση ονομάτων labels (balanced)
label_map_path = VEC / "label_map_balanced.json"
with open(label_map_path, "r") as f:
    label_names = json.load(f)["labels"]

n_labels = len(label_names)
print("Loaded balanced labels:", n_labels)

# Χρησιμοποιούμε ΜΗ capped X + balanced Y, όπως στο 03_train_eval_capped.ipynb
def non_capped_train_test_parts():
    # train_part*.npz / test_part*.npz χωρίς _capped
    train_X = sort_by_index([p for p in VEC.glob("train_part*.npz") if "_capped" not in p.name])
    test_X  = sort_by_index([p for p in VEC.glob("test_part*.npz")  if "_capped" not in p.name])

    y_train = sort_by_index(list(VEC.glob("y_train_part_balanced*.npy")))
    y_test  = sort_by_index(list(VEC.glob("y_test_part_balanced*.npy")))
    return train_X, test_X, y_train, y_test

XTRN, XTE, YTRN_BAL, YTE_BAL = non_capped_train_test_parts()

print("Train parts (X):", len(XTRN), "| Test parts (X):", len(XTE))
print("Train parts (Y):", len(YTRN_BAL), "| Test parts (Y):", len(YTE_BAL))

print("\nExample train:", XTRN[0].name, "|", YTRN_BAL[0].name)
print("Example test :", XTE[0].name,  "|", YTE_BAL[0].name)


Loaded balanced labels: 64
Train parts (X): 551 | Test parts (X): 45
Train parts (Y): 551 | Test parts (Y): 45

Example train: train_part000.npz | y_train_part_balanced000.npy
Example test : test_part000.npz | y_test_part_balanced000.npy


**Cell 5: Calculate positives per label & cap_train**

In [ ]:
pos_train = np.zeros(n_labels, dtype=np.int64)
rows_train = 0

for yp in YTRN_BAL:
    y = np.load(yp, allow_pickle=False)  # (rows, n_labels) int8
    pos_train += y.sum(axis=0).astype(np.int64)
    rows_train += y.shape[0]

min_pos = int(pos_train[pos_train > 0].min())
cap_train = 100 * max(1, min_pos)   # "no more than 100× of the min" στο TRAIN

print("Total train rows:", rows_train)
print("Min positives per label:", min_pos, "| cap_train:", cap_train)


Total train rows: 2754289
Min positives per label: 725 | cap_train: 72500


**Cell 6: RNG, sampling parameters & plan_per_part_for_label**

In [17]:
# Σταθερό RNG για επαναληψιμότητα
rng = np.random.default_rng(12345)

# Πόσα negatives ανά positive στο per-label dataset
NEG_POS_RATIO = 3     # π.χ. 3:1
# Ποσοστό του per-label dataset που κρατάμε για validation
VAL_FRACTION = 0.10   # 10%

def plan_per_part_for_label(label_idx: int, cap_pos: int):
    """
    Επιστρέφει vector με μήκος = #train parts.
    Για κάθε part λέει πόσα POS (δείγματα με y=1) θα κρατήσουμε για αυτό το label,
    έτσι ώστε συνολικά να μην ξεπεράσουμε το cap_pos.
    """
    pos_counts = []
    for yp in YTRN_BAL:
        y = np.load(yp, allow_pickle=False)  # (rows, n_labels)
        pos_counts.append(int(y[:, label_idx].sum()))
    pos_counts = np.array(pos_counts, dtype=np.int64)

    total_pos = int(pos_counts.sum())
    keep_total = min(total_pos, cap_pos)

    if total_pos == 0:
        # αν δεν υπάρχει ποτέ θετικό για αυτό το label
        return np.zeros(len(YTRN_BAL), dtype=np.int64)

    # αναλογική κατανομή ως προς το πόσα POS έχει κάθε part
    frac = pos_counts / total_pos
    plan = np.floor(frac * keep_total).astype(np.int64)

    # αν λόγω floor λείπουν μερικά POS, τα μοιράζουμε στα "πιο πλούσια" parts
    short = int(keep_total - plan.sum())
    if short > 0:
        order = np.argsort(-(frac - plan / keep_total))
        for i in order[:short]:
            plan[i] += 1

    return plan  # ανά part πόσα POS


**Cell 7: Building dataset & LightGBM training per label**

In [18]:
# Βάση για τα μοντέλα ablation – αργότερα μπορούμε να αλλάζουμε υποφακέλους ανά experiment
MODELS_DIR = VEC / "models_lgb_ablation"
MODELS_DIR.mkdir(exist_ok=True)

THRESH_PATH = VEC / "per_label_thresholds_ablation.json"

def model_path_for(label_idx: int) -> Path:
    return MODELS_DIR / f"lgb_label_{label_idx:03d}.txt"


def build_dataset_for_label(label_idx: int,
                            pos_plan_per_part,
                            feature_cols: np.ndarray | None = None):
    """
    Φτιάχνει per-label dataset με capping:
    - Για κάθε train part:
        * παίρνει k_pos POS (σύμφωνα με pos_plan_per_part)
        * παίρνει μέχρι NEG_POS_RATIO * k_pos NEG
    - Κάνει shuffle και split σε train/val (με VAL_FRACTION).
    - Αν δοθεί feature_cols, κρατάει μόνο αυτές τις στήλες.
    """
    X_list, y_list = [], []

    for pi, (xp, yp) in enumerate(zip(XTRN, YTRN_BAL)):
        Y = np.load(yp, allow_pickle=False)        # (rows, n_labels)
        X = sp.load_npz(xp)                        # csr matrix

        y_col = Y[:, label_idx].astype(np.int8)
        pos_idx = np.flatnonzero(y_col == 1)
        neg_idx = np.flatnonzero(y_col == 0)

        # πόσα POS απ' αυτό το part;
        k_pos = int(pos_plan_per_part[pi])
        if k_pos > 0 and len(pos_idx) > 0:
            pos_pick = rng.choice(pos_idx, size=min(k_pos, len(pos_idx)), replace=False)
        else:
            pos_pick = np.array([], dtype=np.int64)

        # πόσα NEG θα πάρουμε από αυτό το part;
        k_neg = int(min(len(neg_idx), NEG_POS_RATIO * max(1, len(pos_pick))))
        if k_neg > 0:
            neg_pick = rng.choice(neg_idx, size=k_neg, replace=False)
        else:
            neg_pick = np.array([], dtype=np.int64)

        pick = np.concatenate([pos_pick, neg_pick])
        if pick.size == 0:
            # τίποτα χρήσιμο από αυτό το part για αυτό το label
            del X, Y
            continue

        # subset rows (και columns αν κάνουμε ablation)
        X_part = X[pick]
        if feature_cols is not None:
            X_part = X_part[:, feature_cols]

        y_part = y_col[pick]

        X_list.append(X_part)
        y_list.append(y_part)

        del X, Y
        gc.collect()

    if not X_list:
        # κανένα δείγμα για αυτό το label
        return None, None, None, None

    X_all = sp.vstack(X_list).tocsr()
    y_all = np.concatenate(y_list).astype(np.int8)

    # shuffle
    order = rng.permutation(X_all.shape[0])
    X_all = X_all[order]
    y_all = y_all[order]

    # split σε train / val
    n_val = int(ceil(VAL_FRACTION * X_all.shape[0]))
    X_val = X_all[:n_val]
    y_val = y_all[:n_val]
    X_tr  = X_all[n_val:]
    y_tr  = y_all[n_val:]

    return X_tr, y_tr, X_val, y_val


def train_one_label(label_idx: int,
                    cap_pos: int,
                    feature_cols: np.ndarray | None = None):
    """
    Κάνει όλη τη διαδικασία για ένα label:
    - Υπολογίζει pos plan ανά part
    - Χτίζει per-label dataset (train/val)
    - Τρέχει LightGBM
    - Κάνει threshold tuning στο validation (F1)
    - Σώζει το μοντέλο & επιστρέφει το best threshold
    """
    name = label_names[label_idx]
    plan = plan_per_part_for_label(label_idx, cap_pos)

    X_tr, y_tr, X_val, y_val = build_dataset_for_label(label_idx, plan, feature_cols)
    if X_tr is None:
        print(f"[{label_idx:03d}] '{name}': no data — skipped")
        return None

    params = {
        "objective": "binary",
        "metric": "auc",
        "learning_rate": 0.05,
        "num_leaves": 31,
        "max_bin": 63,
        "feature_fraction": 0.7,
        "bagging_fraction": 0.7,
        "bagging_freq": 1,
        "min_data_in_leaf": 50,
        "min_sum_hessian_in_leaf": 1e-3,
        "num_threads": 0,
        "verbose": -1,
        "seed": 2025,
    }

    dtrain = lgb.Dataset(X_tr, label=y_tr, free_raw_data=True)
    dvalid = lgb.Dataset(X_val, label=y_val, reference=dtrain, free_raw_data=True)

    booster = lgb.train(
        params,
        dtrain,
        num_boost_round=200,
        valid_sets=[dvalid],
        valid_names=["valid"],
        callbacks=[
            lgb.early_stopping(stopping_rounds=20),
            lgb.log_evaluation(period=25),
        ],
    )

    # Threshold tuning στο validation: βρες threshold που μεγιστοποιεί F1
    val_prob = booster.predict(X_val, num_iteration=booster.best_iteration)
    thr_grid = np.linspace(0.05, 0.95, 19)
    best_thr, best_f1 = 0.5, -1.0

    for t in thr_grid:
        y_hat = (val_prob >= t).astype(np.int8)
        tp = ((y_hat == 1) & (y_val == 1)).sum()
        fp = ((y_hat == 1) & (y_val == 0)).sum()
        fn = ((y_hat == 0) & (y_val == 1)).sum()
        denom = (2 * tp + fp + fn)
        f1 = 0.0 if denom == 0 else (2 * tp) / denom

        if f1 > best_f1:
            best_f1, best_thr = f1, float(t)

    booster.save_model(str(model_path_for(label_idx)))
    print(
        f"[{label_idx:03d}] '{name}'  "
        f"best_iter={booster.best_iteration}  "
        f"best_thr={best_thr:.2f}  val_F1={best_f1:.3f}"
    )

    return best_thr


**Cell 8: Looking for all the .json files**

In [ ]:
meta_files = list(VEC.glob("*.json"))
print(f"Found {len(meta_files)} JSON files in {VEC}:\n")
for f in meta_files:
    print(" -", f.name)

# Φιλτράρουμε όσα μοιάζουν σχετικό με features
feature_meta_files = [f for f in meta_files if "feature" in f.name.lower() or "ember" in f.name.lower()]
print("\nCandidate feature-metadata files:")
for f in feature_meta_files:
    print(" -", f.name)


Found 9 JSON files in /Users/georgektenas/Desktop/Malware Project/data/behavior_vectors_paper:

 - overall_metrics_perlabel_capped.json
 - overall_metrics.json
 - label_map.json
 - label_map_balanced.json
 - overall_metrics_thr.json
 - overall_metrics_capped.json
 - best_thresholds.json
 - per_label_thresholds.json
 - best_thresholds_precision_tuned.json

Candidate feature-metadata files:


**Cell 9: Looking at the content of candidate files**



In [ ]:
# inspect content of candidate feature-metadata JSON
import itertools

for f in feature_meta_files:
    print("\n" + "="*80)
    print("File:", f.name)
    with open(f, "r") as jf:
        data = json.load(jf)

    print("Type:", type(data))

    if isinstance(data, dict):
        keys = list(data.keys())
        print("Top-level keys (up to 10):", keys[:10])

        # αν οι τιμές είναι μικρά dict ή λίστες, δείξε 2-3 παραδείγματα
        sample_items = list(itertools.islice(data.items(), 3))
        for k, v in sample_items:
            print(f"\nKey: {k}")
            print("Value type:", type(v))
            print("Preview:", str(v)[:300], "...")
    elif isinstance(data, list):
        print("List length:", len(data))
        print("First element type:", type(data[0]))
        print("First element preview:", str(data[0])[:300], "...")
    else:
        print("Unknown JSON structure, preview:", str(data)[:300], "...")


**Cell 10: Setting EMBER feature groups (0-based indices, from EMBER spec)**

In [21]:
EMBER_GROUPS = {
    "byte_hist":    np.arange(0,   256),   # 1-256
    "byte_entropy": np.arange(256, 512),   # 257-512
    "strings":      np.arange(512, 616),   # 513-616
    "general":      np.arange(616, 626),   # 617-626
    "header":       np.arange(626, 688),   # 627-688
    "sections":     np.arange(688, 943),   # 689-943
    "imports":      np.arange(943, 2223),  # 944-2223
    "exports":      np.arange(2223, 2351), # 2224-2351
    "data_dirs":    np.arange(2351, 2381), # 2352-2381
}

print("EMBER feature groups:")
total = 0
for name, idxs in EMBER_GROUPS.items():
    print(f" - {name:10s}: start={idxs[0]:4d}, end={idxs[-1]:4d}, len={len(idxs)}")
    total += len(idxs)

print("\nTotal features across all groups:", total)


EMBER feature groups:
 - byte_hist : start=   0, end= 255, len=256
 - byte_entropy: start= 256, end= 511, len=256
 - strings   : start= 512, end= 615, len=104
 - general   : start= 616, end= 625, len=10
 - header    : start= 626, end= 687, len=62
 - sections  : start= 688, end= 942, len=255
 - imports   : start= 943, end=2222, len=1280
 - exports   : start=2223, end=2350, len=128
 - data_dirs : start=2351, end=2380, len=30

Total features across all groups: 2381


**Cell 11: Helper function for ablation experiments**

In [22]:
# Cell 18: Helper συνάρτηση για ablation experiments

def cols_for_groups(group_names):
    """
    Παίρνει λίστα από ονόματα groups (π.χ. ["byte_hist", "imports"])
    και επιστρέφει ΕΝΑ sorted numpy array με όλα τα αντίστοιχα indices.
    """
    arrays = [EMBER_GROUPS[g] for g in group_names]
    cols = np.unique(np.concatenate(arrays))
    return cols

# Sanity check: αν πάρουμε ΟΛΑ τα groups, πρέπει να έχουμε 2381 features 0..2380
all_cols = cols_for_groups(list(EMBER_GROUPS.keys()))
print("All-cols length:", len(all_cols))
print("Min index:", all_cols.min(), "Max index:", all_cols.max())


All-cols length: 2381
Min index: 0 Max index: 2380


**Cell 12: Settings for ablation experiments**

In [23]:
import gc

ABLATION_EXPERIMENTS = [
    {
        "name": "all_features",
        "groups": None,  # None => όλα τα features (baseline)
    },
    {
        "name": "no_imports_exports_dirs",
        "groups": [
            "byte_hist",
            "byte_entropy",
            "strings",
            "general",
            "header",
            "sections",
        ],
    },
    {
        "name": "imports_exports_dirs_only",
        "groups": [
            "imports",
            "exports",
            "data_dirs",
        ],
    },
    {
        "name": "light_static_no_strings_no_imports",
        "groups": [
            "byte_hist",
            "byte_entropy",
            "general",
            "header",
            "sections",
        ],
    },
]

def feature_cols_for_experiment(exp):
    """
    Αν exp["groups"] είναι None -> χρησιμοποιούμε όλα τα features (feature_cols=None).
    Αλλιώς φτιάχνουμε union από τα αντίστοιχα EMBER_GROUPS.
    """
    if exp["groups"] is None:
        return None
    return cols_for_groups(exp["groups"])

print("Defined ablation experiments:")
for exp in ABLATION_EXPERIMENTS:
    g = exp["groups"]
    if g is None:
        print(f" - {exp['name']}: ALL features")
    else:
        cols = feature_cols_for_experiment(exp)
        print(f" - {exp['name']}: groups={g}, n_features={len(cols)}")


Defined ablation experiments:
 - all_features: ALL features
 - no_imports_exports_dirs: groups=['byte_hist', 'byte_entropy', 'strings', 'general', 'header', 'sections'], n_features=943
 - imports_exports_dirs_only: groups=['imports', 'exports', 'data_dirs'], n_features=1438
 - light_static_no_strings_no_imports: groups=['byte_hist', 'byte_entropy', 'general', 'header', 'sections'], n_features=839


**Cell 13: Test ablation run for only label 0**

In [24]:
import time

test_label_idx = 0
test_label_name = label_names[test_label_idx]
print(f"Ablation test for label {test_label_idx}: '{test_label_name}'")

ablation_results_single_label = []

for exp in ABLATION_EXPERIMENTS:
    exp_name = exp["name"]
    cols = feature_cols_for_experiment(exp)

    print("\n" + "="*80)
    print(f"Experiment: {exp_name}")
    if cols is None:
        print("Using ALL features")
    else:
        print(f"Using {len(cols)} features from groups: {exp['groups']}")

    t0 = time.time()
    best_thr = train_one_label(
        label_idx=test_label_idx,
        cap_pos=cap_train,
        feature_cols=cols,
    )
    elapsed_min = (time.time() - t0) / 60.0

    ablation_results_single_label.append({
        "experiment": exp_name,
        "label_idx": test_label_idx,
        "label_name": test_label_name,
        "best_threshold": best_thr,
        "time_min": elapsed_min,
    })

print("\nSummary (single-label ablation):")
for r in ablation_results_single_label:
    print(
        f"{r['experiment']:30s} | "
        f"thr={r['best_threshold']:.2f} | "
        f"time={r['time_min']:.1f} min"
    )


Ablation test for label 0: 'adware'

Experiment: all_features
Using ALL features
Training until validation scores don't improve for 20 rounds
[25]	valid's auc: 0.976026
[50]	valid's auc: 0.981321
[75]	valid's auc: 0.98454
[100]	valid's auc: 0.986348
[125]	valid's auc: 0.987708
[150]	valid's auc: 0.988641
[175]	valid's auc: 0.989266
[200]	valid's auc: 0.98982
Did not meet early stopping. Best iteration is:
[200]	valid's auc: 0.98982
[000] 'adware'  best_iter=200  best_thr=0.50  val_F1=0.914

Experiment: no_imports_exports_dirs
Using 943 features from groups: ['byte_hist', 'byte_entropy', 'strings', 'general', 'header', 'sections']
Training until validation scores don't improve for 20 rounds
[25]	valid's auc: 0.973452
[50]	valid's auc: 0.979195
[75]	valid's auc: 0.982362
[100]	valid's auc: 0.984508
[125]	valid's auc: 0.986057
[150]	valid's auc: 0.987088
[175]	valid's auc: 0.987838
[200]	valid's auc: 0.988478
Did not meet early stopping. Best iteration is:
[200]	valid's auc: 0.988478
[000

**Cell 14: Updated training LightGBM**

In [25]:
# Cell: Updated train_one_label -> επιστρέφει και best_F1

def train_one_label(label_idx: int,
                    cap_pos: int,
                    feature_cols: np.ndarray | None = None):
    """
    Κάνει όλη τη διαδικασία για ένα label:
    - Υπολογίζει pos plan ανά part
    - Χτίζει per-label dataset (train/val)
    - Τρέχει LightGBM
    - Κάνει threshold tuning στο validation (F1)
    - Σώζει το μοντέλο & ΕΠΙΣΤΡΕΦΕΙ (best_threshold, best_val_F1)
    """
    name = label_names[label_idx]
    plan = plan_per_part_for_label(label_idx, cap_pos)

    X_tr, y_tr, X_val, y_val = build_dataset_for_label(label_idx, plan, feature_cols)
    if X_tr is None:
        print(f"[{label_idx:03d}] '{name}': no data — skipped")
        return None, None

    params = {
        "objective": "binary",
        "metric": "auc",
        "learning_rate": 0.05,
        "num_leaves": 31,
        "max_bin": 63,
        "feature_fraction": 0.7,
        "bagging_fraction": 0.7,
        "bagging_freq": 1,
        "min_data_in_leaf": 50,
        "min_sum_hessian_in_leaf": 1e-3,
        "num_threads": 0,
        "verbose": -1,
        "seed": 2025,
    }

    dtrain = lgb.Dataset(X_tr, label=y_tr, free_raw_data=True)
    dvalid = lgb.Dataset(X_val, label=y_val, reference=dtrain, free_raw_data=True)

    booster = lgb.train(
        params,
        dtrain,
        num_boost_round=200,
        valid_sets=[dvalid],
        valid_names=["valid"],
        callbacks=[
            lgb.early_stopping(stopping_rounds=20),
            lgb.log_evaluation(period=50),
        ],
    )

    # Threshold tuning στο validation: βρες threshold που μεγιστοποιεί F1
    val_prob = booster.predict(X_val, num_iteration=booster.best_iteration)
    thr_grid = np.linspace(0.05, 0.95, 19)
    best_thr, best_f1 = 0.5, -1.0

    for t in thr_grid:
        y_hat = (val_prob >= t).astype(np.int8)
        tp = ((y_hat == 1) & (y_val == 1)).sum()
        fp = ((y_hat == 1) & (y_val == 0)).sum()
        fn = ((y_hat == 0) & (y_val == 1)).sum()
        denom = (2 * tp + fp + fn)
        f1 = 0.0 if denom == 0 else (2 * tp) / denom

        if f1 > best_f1:
            best_f1, best_thr = f1, float(t)

    booster.save_model(str(model_path_for(label_idx)))
    print(
        f"[{label_idx:03d}] '{name}'  "
        f"best_iter={booster.best_iteration}  "
        f"best_thr={best_thr:.2f}  val_F1={best_f1:.3f}"
    )

    return best_thr, best_f1


**Cell 15: Helper for full experiment for all labels**

In [26]:
# Cell: Helper για "full experiment" σε όλα τα labels

import time
import pandas as pd

RESULTS_DIR = VEC / "results_ablation"
RESULTS_DIR.mkdir(exist_ok=True)

def run_experiment_all_labels(exp_name: str, group_names):
    """
    Τρέχει ένα ablation experiment για ΟΛΑ τα labels.
    - exp_name: όνομα πειράματος (π.χ. 'light_static_no_strings_no_imports')
    - group_names: λίστα από EMBER_GROUPS keys (ή None για όλα τα features)
    Επιστρέφει pandas DataFrame με τα αποτελέσματα
    και αποθηκεύει CSV στο RESULTS_DIR.
    """
    if group_names is None:
        feature_cols = None
        n_feats = XTRN[0].shape[1]
        desc = f"ALL {n_feats} features"
    else:
        feature_cols = cols_for_groups(group_names)
        n_feats = len(feature_cols)
        desc = f"{n_feats} features from groups {group_names}"

    print("#" * 80)
    print(f"Running experiment '{exp_name}'")
    print(" ", desc)

    all_rows = []
    t0_exp = time.time()

    for li in range(n_labels):
        label_name = label_names[li]
        print(f"\n[{li:02d}/{n_labels-1:02d}] Label {li}: '{label_name}'")

        t0_label = time.time()
        best_thr, best_f1 = train_one_label(
            label_idx=li,
            cap_pos=cap_train,
            feature_cols=feature_cols,
        )
        elapsed_label = (time.time() - t0_label) / 60.0

        row = {
            "experiment": exp_name,
            "groups": None if group_names is None else ",".join(group_names),
            "n_features": n_feats,
            "label_idx": li,
            "label_name": label_name,
            "best_threshold": best_thr,
            "val_F1": best_f1,
            "time_min": elapsed_label,
        }
        all_rows.append(row)

        gc.collect()

    total_min = (time.time() - t0_exp) / 60.0
    print(f"\nExperiment '{exp_name}' finished in {total_min:.1f} minutes.")

    df = pd.DataFrame(all_rows)
    out_csv = RESULTS_DIR / f"ablation_{exp_name}.csv"
    df.to_csv(out_csv, index=False)
    print("Saved results to:", out_csv)

    return df


**Cell 16: EXPERIMENT 1**

In [27]:
#light_static_no_strings_no_imports (839 features)

exp1_name = "light_static_no_strings_no_imports"
exp1_groups = ["byte_hist", "byte_entropy", "general", "header", "sections"]

df_exp1 = run_experiment_all_labels(exp1_name, exp1_groups)
df_exp1.head()


################################################################################
Running experiment 'light_static_no_strings_no_imports'
  839 features from groups ['byte_hist', 'byte_entropy', 'general', 'header', 'sections']

[00/63] Label 0: 'adware'
Training until validation scores don't improve for 20 rounds
[50]	valid's auc: 0.978792
[100]	valid's auc: 0.983886
[150]	valid's auc: 0.986435
[200]	valid's auc: 0.988034
Did not meet early stopping. Best iteration is:
[200]	valid's auc: 0.988034
[000] 'adware'  best_iter=200  best_thr=0.40  val_F1=0.909

[01/63] Label 1: 'antiav'
Training until validation scores don't improve for 20 rounds
[50]	valid's auc: 0.978956
[100]	valid's auc: 0.986479
[150]	valid's auc: 0.989679
[200]	valid's auc: 0.991498
Did not meet early stopping. Best iteration is:
[200]	valid's auc: 0.991498
[001] 'antiav'  best_iter=200  best_thr=0.35  val_F1=0.945

[02/63] Label 2: 'antifw'
Training until validation scores don't improve for 20 rounds
[50]	valid's auc:

,experiment,groups,n_features,label_idx,label_name,best_threshold,val_F1,time_min
0,light_static_no_strings_no_imports,"byte_hist,byte_entropy,general,header,sections",839,0,adware,0.40,0.908793,0.488149
1,light_static_no_strings_no_imports,"byte_hist,byte_entropy,general,header,sections",839,1,antiav,0.35,0.945106,0.507878
2,light_static_no_strings_no_imports,"byte_hist,byte_entropy,general,header,sections",839,2,antifw,0.60,0.995100,0.394165
3,light_static_no_strings_no_imports,"byte_hist,byte_entropy,general,header,sections",839,3,autorun,0.45,0.911220,0.500875
4,light_static_no_strings_no_imports,"byte_hist,byte_entropy,general,header,sections",839,4,backdoor,0.35,0.758691,0.495765


**Cell 17: Overall Micro / Macro / Weighted metrics for EXPERIMENT 1 on VAL SET**

In [28]:
from sklearn.metrics import roc_auc_score
import numpy as np
import pandas as pd

# Ορισμός των groups για το Experiment 1 (όπως πριν)
exp1_groups = ["byte_hist", "byte_entropy", "general", "header", "sections"]
feature_cols_exp1 = cols_for_groups(exp1_groups)

def overall_metrics_from_df(df_results, feature_cols):
    """
    Υπολογίζει συνολικό Precision / Recall / F1 / AUC
    (micro, macro, weighted) πάνω στα validation sets ΚΑΘΕ label,
    χρησιμοποιώντας τα ήδη εκπαιδευμένα LightGBM + thresholds του df_results.
    """

    TP_total = 0
    FP_total = 0
    FN_total = 0

    precisions = []
    recalls = []
    f1s = []
    aucs = []
    supports = []          # #positives ανά label
    all_y = []             # για micro AUC (flatten)
    all_p = []             # για micro AUC

    # Προσοχή: να έχουμε σταθερό RNG για reproducibility
    global rng
    rng = np.random.default_rng(2025)

    for li in range(n_labels):
        name = label_names[li]

        row = df_results.loc[df_results["label_idx"] == li]
        if row.empty:
            print(f"⚠️  Δεν βρέθηκε threshold για label {li} – το αγνοώ.")
            continue

        thr = float(row["best_threshold"].iloc[0])

        # Φτιάχνουμε το dataset για το συγκεκριμένο label (όπως στο training)
        plan = plan_per_part_for_label(li, cap_train)
        X_tr, y_tr, X_val, y_val = build_dataset_for_label(li, plan, feature_cols)

        if X_val is None or y_val is None:
            print(f"⚠️  Label {li} ('{name}') δεν έχει validation δείγματα – skip.")
            continue

        # Φορτώνουμε το ήδη εκπαιδευμένο μοντέλο
        booster = lgb.Booster(model_file=str(model_path_for(li)))
        prob = booster.predict(X_val, num_iteration=booster.best_iteration)
        y_hat = (prob >= thr).astype(np.int8)

        # TP/FP/FN per-label
        tp = int(((y_hat == 1) & (y_val == 1)).sum())
        fp = int(((y_hat == 1) & (y_val == 0)).sum())
        fn = int(((y_hat == 0) & (y_val == 1)).sum())
        tn = int(((y_hat == 0) & (y_val == 0)).sum())

        support = tp + fn  # #positive examples για το label

        # Per-label metrics
        prec = 0.0 if (tp + fp) == 0 else tp / (tp + fp)
        rec  = 0.0 if (tp + fn) == 0 else tp / (tp + fn)
        denom = 2*tp + fp + fn
        f1   = 0.0 if denom == 0 else (2*tp) / denom

        try:
            auc = roc_auc_score(y_val, prob)
        except ValueError:
            auc = np.nan  # αν το label έχει μόνο μία κλάση στο val set

        precisions.append(prec)
        recalls.append(rec)
        f1s.append(f1)
        aucs.append(auc)
        supports.append(support)

        TP_total += tp
        FP_total += fp
        FN_total += fn

        # για micro AUC (flatten όλων των (y, prob))
        all_y.append(y_val.astype(np.int8).ravel())
        all_p.append(prob.ravel())

        gc.collect()

    precisions = np.array(precisions)
    recalls    = np.array(recalls)
    f1s        = np.array(f1s)
    aucs       = np.array(aucs)
    supports   = np.array(supports, dtype=float)

    # ----- Macro -----
    macro_P   = np.nanmean(precisions)
    macro_R   = np.nanmean(recalls)
    macro_F1  = np.nanmean(f1s)
    macro_AUC = np.nanmean(aucs)

    # ----- Weighted (βάρη = #positives ανά label) -----
    w = supports / supports.sum()
    weighted_P   = np.nansum(precisions * w)
    weighted_R   = np.nansum(recalls * w)
    weighted_F1  = np.nansum(f1s * w)
    weighted_AUC = np.nansum(aucs * w)

    # ----- Micro (από συνολικά TP/FP/FN) -----
    micro_P = 0.0 if (TP_total + FP_total) == 0 else TP_total / (TP_total + FP_total)
    micro_R = 0.0 if (TP_total + FN_total) == 0 else TP_total / (TP_total + FN_total)
    denom   = 2*TP_total + FP_total + FN_total
    micro_F1 = 0.0 if denom == 0 else (2*TP_total) / denom

    # Micro AUC: flatten όλων των (y, prob)
    all_y_flat = np.concatenate(all_y)
    all_p_flat = np.concatenate(all_p)
    try:
        micro_AUC = roc_auc_score(all_y_flat, all_p_flat)
    except ValueError:
        micro_AUC = np.nan

    # Φτιάχνουμε ωραίο DataFrame όπως στο screenshot
    table = pd.DataFrame(
        index=["Precision", "Recall", "F1-score", "AUC"],
        columns=["Micro", "Macro", "Weighted"],
        data=[
            [micro_P, macro_P, weighted_P],
            [micro_R, macro_R, weighted_R],
            [micro_F1, macro_F1, weighted_F1],
            [micro_AUC, macro_AUC, weighted_AUC],
        ]
    ).round(4)

    return table

metrics_exp1 = overall_metrics_from_df(df_exp1, feature_cols_exp1)
metrics_exp1


,Micro,Macro,Weighted
Precision,0.9083,0.9242,0.9098
Recall,0.9192,0.9398,0.9192
F1-score,0.9137,0.9313,0.9138
AUC,0.9894,0.9893,0.9845


**Cell 18: Overall metrics on TEST set**

In [ ]:
from sklearn.metrics import precision_recall_fscore_support, roc_auc_score

def overall_test_metrics_for_experiment(df_exp, feature_cols, exp_name):
    # 1) πάρε per-label thresholds από το df_exp (από το ablation run)
    thr_map = {
        int(row["label_idx"]): float(row["best_threshold"])
        for _, row in df_exp.iterrows()
    }

    true_all, prob_all, pred_all = [], [], []

    for pi, (xp, yp) in enumerate(zip(XTE, YTE_BAL)):
        print(f"[test:{exp_name}] part {pi+1}/{len(XTE)} ...")
        X_full = sp.load_npz(xp)
        Y = np.load(yp, allow_pickle=False).astype(np.int8)

        # κρατάμε μόνο τις στήλες του experiment (π.χ. 839 features)
        X = X_full[:, feature_cols]

        probs = np.zeros((X.shape[0], n_labels), dtype=np.float32)
        preds = np.zeros((X.shape[0], n_labels), dtype=np.int8)

           
    for li in range(n_labels):
        # Φορτώνουμε το μοντέλο του συγκεκριμένου label από τον ίδιο φάκελο
        # που χρησιμοποίησε η train_one_label / model_path_for(li)
        model_path = model_path_for(li)
        booster = lgb.Booster(model_file=str(model_path))

        # καλό είναι να χρησιμοποιούμε το best_iteration
        p = booster.predict(X, num_iteration=booster.best_iteration)
        probs[:, li] = p

        thr = thr_map.get(li, 0.5)
        preds[:, li] = (p >= thr).astype(np.int8)

        true_all.append(Y)
        prob_all.append(probs)
        pred_all.append(preds)

    y_true = np.vstack(true_all)
    y_prob = np.vstack(prob_all)
    y_pred = np.vstack(pred_all)

    # Micro / Macro / Weighted Precision–Recall–F1 (όπως στο 03_train_eval_capped)
    micro_P, micro_R, micro_F1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="micro", zero_division=0
    )
    macro_P, macro_R, macro_F1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0
    )
    weighted_P, weighted_R, weighted_F1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="weighted", zero_division=0
    )

    try:
        micro_AUC = roc_auc_score(y_true, y_prob, average="micro")
    except ValueError:
        micro_AUC = np.nan
    try:
        macro_AUC = roc_auc_score(y_true, y_prob, average="macro")
    except ValueError:
        macro_AUC = np.nan
    try:
        weighted_AUC = roc_auc_score(y_true, y_prob, average="weighted")
    except ValueError:
        weighted_AUC = np.nan

    table = pd.DataFrame(
        [
            [micro_P,  macro_P,  weighted_P],
            [micro_R,  macro_R,  weighted_R],
            [micro_F1, macro_F1, weighted_F1],
            [micro_AUC, macro_AUC, weighted_AUC],
        ],
        index=["Precision", "Recall", "F1-score", "AUC"],
        columns=["Micro", "Macro", "Weighted"],
    ).round(4)

    out_csv = DATA_ROOT / "results_ablation" / f"overall_test_metrics_{exp_name}.csv"
    table.to_csv(out_csv)
    print("Saved TEST metrics ->", out_csv)

    return table


In [33]:
test_metrics_exp1 = overall_test_metrics_for_experiment(df_exp1, feature_cols_exp1, exp1_name)
test_metrics_exp1


[test:light_static_no_strings_no_imports] part 1/45 ...
[test:light_static_no_strings_no_imports] part 2/45 ...
[test:light_static_no_strings_no_imports] part 3/45 ...
[test:light_static_no_strings_no_imports] part 4/45 ...
[test:light_static_no_strings_no_imports] part 5/45 ...
[test:light_static_no_strings_no_imports] part 6/45 ...
[test:light_static_no_strings_no_imports] part 7/45 ...
[test:light_static_no_strings_no_imports] part 8/45 ...
[test:light_static_no_strings_no_imports] part 9/45 ...
[test:light_static_no_strings_no_imports] part 10/45 ...
[test:light_static_no_strings_no_imports] part 11/45 ...
[test:light_static_no_strings_no_imports] part 12/45 ...
[test:light_static_no_strings_no_imports] part 13/45 ...
[test:light_static_no_strings_no_imports] part 14/45 ...
[test:light_static_no_strings_no_imports] part 15/45 ...
[test:light_static_no_strings_no_imports] part 16/45 ...
[test:light_static_no_strings_no_imports] part 17/45 ...
[test:light_static_no_strings_no_imports

/opt/homebrew/Caskroom/miniforge/base/envs/maltrain/lib/python3.12/site-packages/sklearn/metrics/_ranking.py:424: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/opt/homebrew/Caskroom/miniforge/base/envs/maltrain/lib/python3.12/site-packages/sklearn/metrics/_ranking.py:424: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/opt/homebrew/Caskroom/miniforge/base/envs/maltrain/lib/python3.12/site-packages/sklearn/metrics/_ranking.py:424: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/opt/homebrew/Caskroom/miniforge/base/envs/maltrain/lib/python3.12/site-packages/sklearn/metrics/_ranking.py:424: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/opt/homebrew/Caskroom/miniforge/base/envs/maltrain/lib/pyth

Saved TEST metrics -> /Users/georgektenas/Desktop/Malware Project/data/behavior_vectors_paper/results_ablation/overall_test_metrics_light_static_no_strings_no_imports.csv


/opt/homebrew/Caskroom/miniforge/base/envs/maltrain/lib/python3.12/site-packages/sklearn/metrics/_ranking.py:424: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/opt/homebrew/Caskroom/miniforge/base/envs/maltrain/lib/python3.12/site-packages/sklearn/metrics/_ranking.py:424: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(


,Micro,Macro,Weighted
Precision,0.0991,0.0934,0.3903
Recall,0.2435,0.2630,0.2435
F1-score,0.1409,0.0933,0.2548
AUC,0.8063,NaN,0.6288


**Cell 19: Training for EXPERIMENT 2**

In [34]:
# κρατάμε όλα τα groups εκτός από header + sections
# groups: byte_hist, byte_entropy, general, strings, imports, exports, data_dirs

exp2_name = "all_except_header_sections"
exp2_groups = [
    "byte_hist",
    "byte_entropy",
    "general",
    "strings",
    "imports",
    "exports",
    "data_dirs",
]

# Χρησιμοποιούμε τον ίδιο helper όπως στο Πείραμα 1
feature_cols_exp2 = cols_for_groups(exp2_groups)

df_exp2 = run_experiment_all_labels(exp2_name, exp2_groups)
df_exp2.head()


################################################################################
Running experiment 'all_except_header_sections'
  2064 features from groups ['byte_hist', 'byte_entropy', 'general', 'strings', 'imports', 'exports', 'data_dirs']

[00/63] Label 0: 'adware'
Training until validation scores don't improve for 20 rounds
[50]	valid's auc: 0.980137
[100]	valid's auc: 0.9855
[150]	valid's auc: 0.987933
[200]	valid's auc: 0.989254
Did not meet early stopping. Best iteration is:
[200]	valid's auc: 0.989254
[000] 'adware'  best_iter=200  best_thr=0.45  val_F1=0.912

[01/63] Label 1: 'antiav'
Training until validation scores don't improve for 20 rounds
[50]	valid's auc: 0.9754
[100]	valid's auc: 0.984256
[150]	valid's auc: 0.988111
[200]	valid's auc: 0.989912
Did not meet early stopping. Best iteration is:
[200]	valid's auc: 0.989912
[001] 'antiav'  best_iter=200  best_thr=0.35  val_F1=0.939

[02/63] Label 2: 'antifw'
Training until validation scores don't improve for 20 rounds
Earl

,experiment,groups,n_features,label_idx,label_name,best_threshold,val_F1,time_min
0,all_except_header_sections,"byte_hist,byte_entropy,general,strings,imports...",2064,0,adware,0.45,0.912250,0.611323
1,all_except_header_sections,"byte_hist,byte_entropy,general,strings,imports...",2064,1,antiav,0.35,0.939370,0.588880
2,all_except_header_sections,"byte_hist,byte_entropy,general,strings,imports...",2064,2,antifw,0.30,0.996221,0.409259
3,all_except_header_sections,"byte_hist,byte_entropy,general,strings,imports...",2064,3,autorun,0.40,0.922543,0.605301
4,all_except_header_sections,"byte_hist,byte_entropy,general,strings,imports...",2064,4,backdoor,0.35,0.764314,0.593944


**Cell 20: Metrics for VAL set EXP 2**

In [35]:
# VAL metrics για το Experiment 2 (all_except_header_sections)

metrics_exp2_val = overall_metrics_from_df(df_exp2, feature_cols_exp2)
metrics_exp2_val


,Micro,Macro,Weighted
Precision,0.9062,0.9225,0.9083
Recall,0.9249,0.9436,0.9249
F1-score,0.9155,0.9322,0.9158
AUC,0.9882,0.9895,0.9850


**Cell 21: Metrics for TEST set EXP 2**

In [36]:
# TEST metrics για το Experiment 2 (all_except_header_sections)

test_metrics_exp2 = overall_test_metrics_for_experiment(
    df_exp2,
    feature_cols_exp2,
    exp2_name,        # "all_except_header_sections"
)
test_metrics_exp2


[test:all_except_header_sections] part 1/45 ...
[test:all_except_header_sections] part 2/45 ...
[test:all_except_header_sections] part 3/45 ...
[test:all_except_header_sections] part 4/45 ...
[test:all_except_header_sections] part 5/45 ...
[test:all_except_header_sections] part 6/45 ...
[test:all_except_header_sections] part 7/45 ...
[test:all_except_header_sections] part 8/45 ...
[test:all_except_header_sections] part 9/45 ...
[test:all_except_header_sections] part 10/45 ...
[test:all_except_header_sections] part 11/45 ...
[test:all_except_header_sections] part 12/45 ...
[test:all_except_header_sections] part 13/45 ...
[test:all_except_header_sections] part 14/45 ...
[test:all_except_header_sections] part 15/45 ...
[test:all_except_header_sections] part 16/45 ...
[test:all_except_header_sections] part 17/45 ...
[test:all_except_header_sections] part 18/45 ...
[test:all_except_header_sections] part 19/45 ...
[test:all_except_header_sections] part 20/45 ...
[test:all_except_header_secti

/opt/homebrew/Caskroom/miniforge/base/envs/maltrain/lib/python3.12/site-packages/sklearn/metrics/_ranking.py:424: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/opt/homebrew/Caskroom/miniforge/base/envs/maltrain/lib/python3.12/site-packages/sklearn/metrics/_ranking.py:424: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/opt/homebrew/Caskroom/miniforge/base/envs/maltrain/lib/python3.12/site-packages/sklearn/metrics/_ranking.py:424: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/opt/homebrew/Caskroom/miniforge/base/envs/maltrain/lib/python3.12/site-packages/sklearn/metrics/_ranking.py:424: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/opt/homebrew/Caskroom/miniforge/base/envs/maltrain/lib/pyth

Saved TEST metrics -> /Users/georgektenas/Desktop/Malware Project/data/behavior_vectors_paper/results_ablation/overall_test_metrics_all_except_header_sections.csv


/opt/homebrew/Caskroom/miniforge/base/envs/maltrain/lib/python3.12/site-packages/sklearn/metrics/_ranking.py:424: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/opt/homebrew/Caskroom/miniforge/base/envs/maltrain/lib/python3.12/site-packages/sklearn/metrics/_ranking.py:424: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/opt/homebrew/Caskroom/miniforge/base/envs/maltrain/lib/python3.12/site-packages/sklearn/metrics/_ranking.py:424: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(


,Micro,Macro,Weighted
Precision,0.1351,0.1115,0.4614
Recall,0.2859,0.2478,0.2859
F1-score,0.1835,0.1121,0.2752
AUC,0.8443,NaN,0.7461


**Cell 22: Training for EXPERIMENT 3**

In [37]:
# κρατάμε όλα εκτός από byte_hist & byte_entropy
# groups: header, sections, general, strings, imports, exports, data_dirs

exp3_name = "all_except_bytehist_entropy"
exp3_groups = [
    "header",
    "sections",
    "general",
    "strings",
    "imports",
    "exports",
    "data_dirs",
]

# Ποιες στήλες features αντιστοιχούν σε αυτά τα groups
feature_cols_exp3 = cols_for_groups(exp3_groups)

# Τρέχουμε LightGBM για ΟΛΑ τα labels, όπως στα προηγούμενα πειράματα
df_exp3 = run_experiment_all_labels(exp3_name, exp3_groups)
df_exp3.head()


################################################################################
Running experiment 'all_except_bytehist_entropy'
  1869 features from groups ['header', 'sections', 'general', 'strings', 'imports', 'exports', 'data_dirs']

[00/63] Label 0: 'adware'
Training until validation scores don't improve for 20 rounds
[50]	valid's auc: 0.97985
[100]	valid's auc: 0.985526
[150]	valid's auc: 0.987968
[200]	valid's auc: 0.989282
Did not meet early stopping. Best iteration is:
[200]	valid's auc: 0.989282
[000] 'adware'  best_iter=200  best_thr=0.45  val_F1=0.912

[01/63] Label 1: 'antiav'
Training until validation scores don't improve for 20 rounds
[50]	valid's auc: 0.979715
[100]	valid's auc: 0.986518
[150]	valid's auc: 0.989962
[200]	valid's auc: 0.991709
Did not meet early stopping. Best iteration is:
[200]	valid's auc: 0.991709
[001] 'antiav'  best_iter=200  best_thr=0.40  val_F1=0.945

[02/63] Label 2: 'antifw'
Training until validation scores don't improve for 20 rounds
Early s

,experiment,groups,n_features,label_idx,label_name,best_threshold,val_F1,time_min
0,all_except_bytehist_entropy,"header,sections,general,strings,imports,export...",1869,0,adware,0.45,0.911600,0.559333
1,all_except_bytehist_entropy,"header,sections,general,strings,imports,export...",1869,1,antiav,0.40,0.944695,0.545680
2,all_except_bytehist_entropy,"header,sections,general,strings,imports,export...",1869,2,antifw,0.45,0.996221,0.455234
3,all_except_bytehist_entropy,"header,sections,general,strings,imports,export...",1869,3,autorun,0.40,0.927748,0.558176
4,all_except_bytehist_entropy,"header,sections,general,strings,imports,export...",1869,4,backdoor,0.35,0.776105,0.544342


**Cell 23: Metrics for VAL set EXP 3**

In [ ]:
# VAL metrics για το Experiment 3

metrics_exp3_val = overall_metrics_from_df(df_exp3, feature_cols_exp3)
metrics_exp3_val


,Micro,Macro,Weighted
Precision,0.9128,0.9270,0.9146
Recall,0.9277,0.9467,0.9277
F1-score,0.9202,0.9361,0.9203
AUC,0.9907,0.9905,0.9864


**Cell 24: Metrics for TEST set EXP 3**

In [39]:
# TEST metrics για το Experiment 3 

test_metrics_exp3 = overall_test_metrics_for_experiment(
    df_exp3,
    feature_cols_exp3,
    exp3_name,
)
test_metrics_exp3


[test:all_except_bytehist_entropy] part 1/45 ...
[test:all_except_bytehist_entropy] part 2/45 ...
[test:all_except_bytehist_entropy] part 3/45 ...
[test:all_except_bytehist_entropy] part 4/45 ...
[test:all_except_bytehist_entropy] part 5/45 ...
[test:all_except_bytehist_entropy] part 6/45 ...
[test:all_except_bytehist_entropy] part 7/45 ...
[test:all_except_bytehist_entropy] part 8/45 ...
[test:all_except_bytehist_entropy] part 9/45 ...
[test:all_except_bytehist_entropy] part 10/45 ...
[test:all_except_bytehist_entropy] part 11/45 ...
[test:all_except_bytehist_entropy] part 12/45 ...
[test:all_except_bytehist_entropy] part 13/45 ...
[test:all_except_bytehist_entropy] part 14/45 ...
[test:all_except_bytehist_entropy] part 15/45 ...
[test:all_except_bytehist_entropy] part 16/45 ...
[test:all_except_bytehist_entropy] part 17/45 ...
[test:all_except_bytehist_entropy] part 18/45 ...
[test:all_except_bytehist_entropy] part 19/45 ...
[test:all_except_bytehist_entropy] part 20/45 ...
[test:all

/opt/homebrew/Caskroom/miniforge/base/envs/maltrain/lib/python3.12/site-packages/sklearn/metrics/_ranking.py:424: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/opt/homebrew/Caskroom/miniforge/base/envs/maltrain/lib/python3.12/site-packages/sklearn/metrics/_ranking.py:424: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/opt/homebrew/Caskroom/miniforge/base/envs/maltrain/lib/python3.12/site-packages/sklearn/metrics/_ranking.py:424: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/opt/homebrew/Caskroom/miniforge/base/envs/maltrain/lib/python3.12/site-packages/sklearn/metrics/_ranking.py:424: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/opt/homebrew/Caskroom/miniforge/base/envs/maltrain/lib/pyth

Saved TEST metrics -> /Users/georgektenas/Desktop/Malware Project/data/behavior_vectors_paper/results_ablation/overall_test_metrics_all_except_bytehist_entropy.csv


/opt/homebrew/Caskroom/miniforge/base/envs/maltrain/lib/python3.12/site-packages/sklearn/metrics/_ranking.py:424: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/opt/homebrew/Caskroom/miniforge/base/envs/maltrain/lib/python3.12/site-packages/sklearn/metrics/_ranking.py:424: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(


,Micro,Macro,Weighted
Precision,0.0812,0.1432,0.5364
Recall,0.2673,0.2807,0.2673
F1-score,0.1246,0.1312,0.2755
AUC,0.8512,NaN,0.8218
